In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
# Configure plotting style
sns.set(style='whitegrid')
import sys
sys.path.append("../scripts")


In [2]:
from loader import load_data
import pandas as pd

df = load_data("../data/MachineLearningRating_v3.txt")

load_data loaded


/home/sasa/Documents/code/KIAM/week3/notebook/../scripts/loader.py:5: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep='|')


In [3]:
#Shape, types, nulls, duplicates
from data_preparation import MyEDA

MyEDA.analyze_data(df)

Shape: (1000098, 52)
Types:
 UnderwrittenCoverID           int64
PolicyID                      int64
TransactionMonth             object
IsVATRegistered                bool
Citizenship                  object
LegalType                    object
Title                        object
Language                     object
Bank                         object
AccountType                  object
MaritalStatus                object
Gender                       object
Country                      object
Province                     object
PostalCode                    int64
MainCrestaZone               object
SubCrestaZone                object
ItemType                     object
mmcode                      float64
VehicleType                  object
RegistrationYear              int64
make                         object
Model                        object
Cylinders                   float64
cubiccapacity               float64
kilowatts                   float64
bodytype                     object

### Caclulate Claim Frequency and add it as column in df

In [3]:
# Calculate 'Claim Frequency': 1 if TotalClaims > 0, else 0
df['ClaimFrequency'] = (df['TotalClaims'] > 0).astype(int)

# You can check the first few rows with the new column
print(df[['TotalClaims', 'ClaimFrequency']].head())

# And see the distribution
print(df['ClaimFrequency'].value_counts())

   TotalClaims  ClaimFrequency
0          0.0               0
1          0.0               0
2          0.0               0
3          0.0               0
4          0.0               0
ClaimFrequency
0    997310
1      2788
Name: count, dtype: int64


### ***calcutate claim severity and print average***

In [4]:
import pandas as pd


# 1. Filter for policies where a claim actually occurred (TotalClaims > 0)
claims_occurred_df = df[df['TotalClaims'] > 0]

# 2. Calculate the average of 'TotalClaims' for these filtered policies
claim_severity = claims_occurred_df['TotalClaims'].mean()

print(f"The overall Claim Severity (average claim amount when a claim occurred) is: ${claim_severity:,.2f}")

The overall Claim Severity (average claim amount when a claim occurred) is: $23,273.39


### ***Calcuate margin and store it in df as column***


In [9]:
import pandas as pd

# Assuming your DataFrame is named 'df' and 'TotalPremium' and 'TotalClaims' columns exist

# Calculate 'Margin' for each policy
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

# Check the first few rows with the new column
print(df[['TotalPremium', 'TotalClaims', 'Margin']].head())

# You might also want to see some summary statistics for Margin
print("\nMargin Summary Statistics:")
print(df['Margin'].describe())

   TotalPremium  TotalClaims      Margin
0     21.929825          0.0   21.929825
1     21.929825          0.0   21.929825
2      0.000000          0.0    0.000000
3    512.848070          0.0  512.848070
4      0.000000          0.0    0.000000

Margin Summary Statistics:
count    1.000098e+06
mean    -2.955694e+00
std      2.367137e+03
min     -3.928486e+05
25%      0.000000e+00
50%      2.157687e+00
75%      2.192982e+01
max      6.528260e+04
Name: Margin, dtype: float64


### Testing Claim Frequency across Provinces 

In [5]:
import pandas as pd
from scipy.stats import chi2_contingency

# Assuming 'df' is your DataFrame with 'Province' and 'ClaimFrequency' columns

print("--- Testing Claim Frequency across Provinces ---")

# 1. Create a Contingency Table
contingency_table_cf = pd.crosstab(df['Province'], df['ClaimFrequency'])
print("Contingency Table (Province vs. ClaimFrequency):\n", contingency_table_cf)

# 2. Perform the Chi-squared Test
chi2, p_value_cf, dof, expected = chi2_contingency(contingency_table_cf)

print(f"\nChi-squared Statistic: {chi2:.4f}")
print(f"P-value: {p_value_cf:.4f}")
print(f"Degrees of Freedom: {dof}")

# 3. Analyze the p-value
alpha = 0.05
if p_value_cf < alpha:
    print(f"\nResult: Reject the Null Hypothesis (p < {alpha}).")
    print("Conclusion: There is a statistically significant difference in Claim Frequency across provinces.")
else:
    print(f"\nResult: Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Conclusion: There is no statistically significant difference in Claim Frequency across provinces.")

# Optional: To see claim frequency percentages by province for better interpretation
province_claim_freq = df.groupby('Province')['ClaimFrequency'].value_counts(normalize=True).unstack()
print("\nClaim Frequency Percentage by Province:")
print(province_claim_freq)

# If the null hypothesis is rejected, you can inspect which provinces have higher/lower frequencies
if p_value_cf < alpha:
    # Example: Calculate actual claim frequency for each province
    actual_claim_rates = df.groupby('Province')['ClaimFrequency'].mean()
    print("\nActual Claim Frequency Rates by Province:")
    print(actual_claim_rates.sort_values(ascending=False))

--- Testing Claim Frequency across Provinces ---
Contingency Table (Province vs. ClaimFrequency):
 ClaimFrequency       0     1
Province                    
Eastern Cape     30286    50
Free State        8088    11
Gauteng         392543  1322
KwaZulu-Natal   169298   483
Limpopo          24769    67
Mpumalanga       52590   128
North West      142938   349
Northern Cape     6372     8
Western Cape    170426   370

Chi-squared Statistic: 104.1909
P-value: 0.0000
Degrees of Freedom: 8

Result: Reject the Null Hypothesis (p < 0.05).
Conclusion: There is a statistically significant difference in Claim Frequency across provinces.

Claim Frequency Percentage by Province:
ClaimFrequency         0         1
Province                          
Eastern Cape    0.998352  0.001648
Free State      0.998642  0.001358
Gauteng         0.996644  0.003356
KwaZulu-Natal   0.997155  0.002845
Limpopo         0.997302  0.002698
Mpumalanga      0.997572  0.002428
North West      0.997564  0.002436
Northern C

In [6]:
import pandas as pd
from scipy.stats import f_oneway

# Assuming 'df' is your DataFrame with 'Province' and 'TotalClaims' columns
# And you've already created the 'Margin' column as well

print("\n--- Testing Claim Severity across Provinces ---")

# 1. Filter data for policies where claims occurred
claims_occurred_df = df[df['TotalClaims'] > 0].copy() # Use .copy() to avoid SettingWithCopyWarning

# 2. Prepare data for ANOVA: Create a list of claim amounts for each province
# Get unique provinces
provinces = claims_occurred_df['Province'].unique()

# Create a list of arrays, where each array contains the TotalClaims for a specific province
# This is required format for f_oneway
claim_amounts_by_province = [claims_occurred_df['TotalClaims'][claims_occurred_df['Province'] == p].values for p in provinces]

# Optional: Print mean claim severity by province for initial inspection
print("Mean Claim Severity by Province (for policies with claims):")
print(claims_occurred_df.groupby('Province')['TotalClaims'].mean().sort_values(ascending=False))


# 3. Perform ANOVA test
f_statistic_cs, p_value_cs = f_oneway(*claim_amounts_by_province)

print(f"\nANOVA F-statistic (Claim Severity): {f_statistic_cs:.4f}")
print(f"P-value (Claim Severity): {p_value_cs:.4f}")

# 4. Analyze the p-value
alpha = 0.05
if p_value_cs < alpha:
    print(f"\nResult: Reject the Null Hypothesis (p < {alpha}).")
    print("Conclusion: There is a statistically significant difference in Claim Severity across provinces.")
else:
    print(f"\nResult: Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Conclusion: There is no statistically significant difference in Claim Severity across provinces.")


--- Testing Claim Severity across Provinces ---
Mean Claim Severity by Province (for policies with claims):
Province
Free State       32265.661085
KwaZulu-Natal    29609.487473
Western Cape     28095.849881
Eastern Cape     27128.533277
Gauteng          22243.878396
North West       16963.467035
Mpumalanga       15979.553421
Limpopo          15171.294187
Northern Cape    11186.313596
Name: TotalClaims, dtype: float64

ANOVA F-statistic (Claim Severity): 4.8302
P-value (Claim Severity): 0.0000

Result: Reject the Null Hypothesis (p < 0.05).
Conclusion: There is a statistically significant difference in Claim Severity across provinces.


In [10]:
import pandas as pd
from scipy.stats import f_oneway

# Assuming 'df' is your DataFrame with 'Province' and 'Margin' columns

print("\n--- Testing Margin across Provinces ---")

# 1. Prepare data for ANOVA: Create a list of Margin amounts for each province
# Get unique provinces
provinces = df['Province'].unique()

# Create a list of arrays, where each array contains the Margin for a specific province
# This is required format for f_oneway
margin_amounts_by_province = [df['Margin'][df['Province'] == p].values for p in provinces]

# Optional: Print mean Margin by province for initial inspection
print("Mean Margin by Province:")
print(df.groupby('Province')['Margin'].mean().sort_values(ascending=False))

# 2. Perform ANOVA test
f_statistic_margin, p_value_margin = f_oneway(*margin_amounts_by_province)

print(f"\nANOVA F-statistic (Margin): {f_statistic_margin:.4f}")
print(f"P-value (Margin): {p_value_margin:.4f}")

# 3. Analyze the p-value
alpha = 0.05
if p_value_margin < alpha:
    print(f"\nResult: Reject the Null Hypothesis (p < {alpha}).")
    print("Conclusion: There is a statistically significant difference in Margin across provinces.")
else:
    print(f"\nResult: Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Conclusion: There is no statistically significant difference in Margin across provinces.")


--- Testing Margin across Provinces ---
Mean Margin by Province:
Province
Northern Cape    35.590527
Eastern Cape     25.833240
Limpopo          20.971484
Free State       20.550805
Mpumalanga       15.016059
North West       10.958832
Western Cape     -3.414689
KwaZulu-Natal    -6.433598
Gauteng         -13.558894
Name: Margin, dtype: float64

ANOVA F-statistic (Margin): 3.2226
P-value (Margin): 0.0011

Result: Reject the Null Hypothesis (p < 0.05).
Conclusion: There is a statistically significant difference in Margin across provinces.


In [16]:
import pandas as pd
from scipy.stats import chi2_contingency

# Assuming 'df' is your DataFrame with 'MainCrestaZone' and 'ClaimFrequency' columns

print("\n--- Testing Claim Frequency across MainCresta Zones ---")

# 1. Create a Contingency Table
contingency_table_cresta_cf = pd.crosstab(df['MainCrestaZone'], df['ClaimFrequency'])
print("Contingency Table (MainCrestaZone vs. ClaimFrequency):\n", contingency_table_cresta_cf.head()) # print head due to potentially more rows

# 2. Perform the Chi-squared Test
chi2_cresta_cf, p_value_cresta_cf, dof_cresta_cf, expected_cresta_cf = chi2_contingency(contingency_table_cresta_cf)

print(f"\nChi-squared Statistic: {chi2_cresta_cf:.4f}")
print(f"P-value: {p_value_cresta_cf:.4f}")
print(f"Degrees of Freedom: {dof_cresta_cf}")

# 3. Analyze the p-value
alpha = 0.05
if p_value_cresta_cf < alpha:
    print(f"\nResult: Reject the Null Hypothesis (p < {alpha}).")
    print("Conclusion: There is a statistically significant difference in Claim Frequency across MainCresta Zones.")
else:
    print(f"\nResult: Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Conclusion: There is no statistically significant difference in Claim Frequency across MainCresta Zones.")

# Optional: To see claim frequency percentages by MainCrestaZone for better interpretation
cresta_claim_freq = df.groupby('MainCrestaZone')['ClaimFrequency'].mean().sort_values(ascending=False)
print("\nActual Claim Frequency Rates by MainCrestaZone (sorted high to low):")
print(cresta_claim_freq)


--- Testing Claim Frequency across MainCresta Zones ---
Contingency Table (MainCrestaZone vs. ClaimFrequency):
 ClaimFrequency                                   0    1
MainCrestaZone                                         
Cape Province                                 6692    9
Cape Province (Cape Town)                    95714  222
Cape Province (East and North of Cape Town)  19341   50
Ciskei, Cape Mid 1                            2410    5
East London                                   3957    4

Chi-squared Statistic: 176.1190
P-value: 0.0000
Degrees of Freedom: 15

Result: Reject the Null Hypothesis (p < 0.05).
Conclusion: There is a statistically significant difference in Claim Frequency across MainCresta Zones.

Actual Claim Frequency Rates by MainCrestaZone (sorted high to low):
MainCrestaZone
Transvaal (Pretoria)                           0.003967
Johannesburg                                   0.003568
Natal (Durban)                                 0.003415
Cape Province (Eas

In [17]:
import pandas as pd
from scipy.stats import f_oneway

# Assuming 'df' is your DataFrame with 'MainCrestaZone' and 'TotalClaims' columns

print("\n--- Testing Claim Severity across MainCresta Zones ---")

# 1. Filter data for policies where claims occurred
claims_occurred_df_cresta = df[df['TotalClaims'] > 0].copy()

# 2. Prepare data for ANOVA: Create a list of claim amounts for each MainCrestaZone
# Get unique MainCrestaZones
cresta_zones = claims_occurred_df_cresta['MainCrestaZone'].unique()

# Create a list of arrays, where each array contains the TotalClaims for a specific MainCrestaZone
claim_amounts_by_cresta_zone = [
    claims_occurred_df_cresta['TotalClaims'][claims_occurred_df_cresta['MainCrestaZone'] == zone].values
    for zone in cresta_zones
]

# Optional: Print mean claim severity by MainCrestaZone for initial inspection
print("Mean Claim Severity by MainCrestaZone (for policies with claims):")
print(claims_occurred_df_cresta.groupby('MainCrestaZone')['TotalClaims'].mean().sort_values(ascending=False))

# 3. Perform ANOVA test
f_statistic_cresta_cs, p_value_cresta_cs = f_oneway(*claim_amounts_by_cresta_zone)

print(f"\nANOVA F-statistic (Claim Severity): {f_statistic_cresta_cs:.4f}")
print(f"P-value (Claim Severity): {p_value_cresta_cs:.4f}")

# 4. Analyze the p-value
alpha = 0.05
if p_value_cresta_cs < alpha:
    print(f"\nResult: Reject the Null Hypothesis (p < {alpha}).")
    print("Conclusion: There is a statistically significant difference in Claim Severity across MainCresta Zones.")
else:
    print(f"\nResult: Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Conclusion: There is no statistically significant difference in Claim Severity across MainCresta Zones.")


--- Testing Claim Severity across MainCresta Zones ---
Mean Claim Severity by MainCrestaZone (for policies with claims):
MainCrestaZone
Port Elizabeth                                 91064.877193
Oranje Free State                              32265.661085
Cape Province (Cape Town)                      31935.325407
Natal (Durban)                                 30483.115283
Natal                                          29135.032948
Cape Province (East and North of Cape Town)    25495.170161
Tembu 2, Cape Mid 2, Cape Mid West, Tembu 1    24660.362345
Johannesburg                                   23032.302601
Rand East                                      22163.699476
Karoo 1 (Northeast of Cape Town)               21753.885422
Transvaal (all except Pretoria)                19214.721685
Transvaal (Pretoria)                           18875.775319
Ciskei, Cape Mid 1                             17135.684211
East London                                    14997.298246
Cape Province          

In [18]:
import pandas as pd
from scipy.stats import f_oneway

# Assuming 'df' is your DataFrame with 'MainCrestaZone' and 'Margin' columns

print("\n--- Testing Margin across MainCresta Zones ---")

# 1. Prepare data for ANOVA: Create a list of Margin amounts for each MainCrestaZone
# Get unique MainCrestaZones
cresta_zones = df['MainCrestaZone'].unique()

# Create a list of arrays, where each array contains the Margin for a specific MainCrestaZone
margin_amounts_by_cresta_zone = [
    df['Margin'][df['MainCrestaZone'] == zone].values
    for zone in cresta_zones
]

# Optional: Print mean Margin by MainCrestaZone for initial inspection
print("Mean Margin by MainCrestaZone:")
print(df.groupby('MainCrestaZone')['Margin'].mean().sort_values(ascending=False))

# 2. Perform ANOVA test
f_statistic_cresta_margin, p_value_cresta_margin = f_oneway(*margin_amounts_by_cresta_zone)

print(f"\nANOVA F-statistic (Margin): {f_statistic_cresta_margin:.4f}")
print(f"P-value (Margin): {p_value_cresta_margin:.4f}")

# 3. Analyze the p-value
alpha = 0.05
if p_value_cresta_margin < alpha:
    print(f"\nResult: Reject the Null Hypothesis (p < {alpha}).")
    print("Conclusion: There is a statistically significant difference in Margin across MainCresta Zones.")
else:
    print(f"\nResult: Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Conclusion: There is no statistically significant difference in Margin across MainCresta Zones.")


--- Testing Margin across MainCresta Zones ---
Mean Margin by MainCrestaZone:
MainCrestaZone
East London                                    73.048714
Langkloof, Coast 2, Coast 1                    50.700608
Ciskei, Cape Mid 1                             45.177758
Cape Province                                  34.574972
Oranje Free State                              21.499101
Tembu 2, Cape Mid 2, Cape Mid West, Tembu 1    20.615455
Karoo 1 (Northeast of Cape Town)               16.331577
Natal                                           8.566809
Transvaal (all except Pretoria)                 6.900011
Rand East                                       3.391847
Port Elizabeth                                 -2.183000
Cape Province (East and North of Cape Town)    -2.344909
Cape Province (Cape Town)                     -15.604794
Transvaal (Pretoria)                          -15.961880
Johannesburg                                  -17.287246
Natal (Durban)                                -23.4

In [20]:
import pandas as pd
from scipy.stats import chi2_contingency

# Assuming 'df' is your DataFrame with 'Gender' and 'ClaimFrequency' columns

print("\n--- Testing Claim Frequency between Women and Men ---")

# 1. Create a Contingency Table
contingency_table_gender_cf = pd.crosstab(df['Gender'], df['ClaimFrequency'])
print("Contingency Table (Gender vs. ClaimFrequency):\n", contingency_table_gender_cf)

# Ensure there are at least two genders for comparison
if len(df['Gender'].unique()) < 2:
    print("\nWarning: Less than two unique gender categories found. Cannot perform comparison.")
else:
    # 2. Perform the Chi-squared Test
    chi2_gender_cf, p_value_gender_cf, dof_gender_cf, expected_gender_cf = chi2_contingency(contingency_table_gender_cf)

    print(f"\nChi-squared Statistic: {chi2_gender_cf:.4f}")
    print(f"P-value: {p_value_gender_cf:.4f}")
    print(f"Degrees of Freedom: {dof_gender_cf}")

    # 3. Analyze the p-value
    alpha = 0.05
    if p_value_gender_cf < alpha:
        print(f"\nResult: Reject the Null Hypothesis (p < {alpha}).")
        print("Conclusion: There is a statistically significant difference in Claim Frequency between Women and Men.")
    else:
        print(f"\nResult: Fail to Reject the Null Hypothesis (p >= {alpha}).")
        print("Conclusion: There is no statistically significant difference in Claim Frequency between Women and Men.")

    # Optional: To see claim frequency percentages by Gender for better interpretation
    gender_claim_freq = df.groupby('Gender')['ClaimFrequency'].mean().sort_values(ascending=False)
    print("\nActual Claim Frequency Rates by Gender:")
    print(gender_claim_freq)


--- Testing Claim Frequency between Women and Men ---
Contingency Table (Gender vs. ClaimFrequency):
 ClaimFrequency       0     1
Gender                      
Female            6741    14
Male             42723    94
Not specified   938324  2666

Chi-squared Statistic: 7.2559
P-value: 0.0266
Degrees of Freedom: 2

Result: Reject the Null Hypothesis (p < 0.05).
Conclusion: There is a statistically significant difference in Claim Frequency between Women and Men.

Actual Claim Frequency Rates by Gender:
Gender
Not specified    0.002833
Male             0.002195
Female           0.002073
Name: ClaimFrequency, dtype: float64
